# S3.10 — Null Handling
**Date completed:** October 2026  
**Status:** Completed  
**Key concept:** NULL == NULL returns NULL — never True

In [0]:
# S3.10 — Cell 2
# NULL Handling:
# isNull(), isNotNull(), fillna(), dropna(), coalesce()
#
# Goal:
# 1. Understand SQL NULL comparison semantics.
# 2. Prove that NULL = NULL does not return True.
# 3. Use isNull() and isNotNull() correctly.
# 4. Understand null-safe equality.
# 5. Use fillna() to replace NULL values.
# 6. Use dropna() to remove rows based on NULL rules.
# 7. Use coalesce() to select the first non-NULL value.
# 8. Apply NULL-handling concepts to Silver-layer data quality.
#
# Environment:
# Databricks Serverless
#
# Important:
# NULL handling must follow business rules.
# Do not replace or remove NULL values blindly.
# ============================================================

import pyspark.sql.functions as F

from pyspark.sql.types import (
    StructType,
    StructField,
    LongType,
    StringType,
    IntegerType,
    DateType,
    DoubleType,
    DecimalType
)

print("=== S3.10: NULL HANDLING IN PYSPARK ===")
print()

In [0]:
# ------------------------------------------------------------
# Step 1 — Create Controlled Data with NULL Values
#
# We use a small controlled DataFrame so that NULL behaviour
# is easy to observe and validate.
#
# NOTE:
# DoubleType is used here only for this NULL-handling exercise.
# For exact financial values, DecimalType may be more appropriate.
# ------------------------------------------------------------

schema = StructType([

    StructField("order_id", IntegerType(), False),

    StructField("customer_id", IntegerType(), True),

    StructField("city", StringType(), True),

    StructField("total_price", DoubleType(), True),

    StructField("status", StringType(), True)])

data = [

    # Complete record
    (1001, 201, "Delhi", 2500.0, "F"),

    # NULL customer_id
    (1002, None, "Mumbai", 1800.0, "O"),

    # NULL city
    (1003, 203, None, 3200.0, "F"),

    # NULL total_price
    (1004, 204, "Chennai", None, "O"),

    # All nullable business attributes are NULL.
    # order_id itself is still populated.
    (1005, None, None, None, None),

    # Complete record
    (1006, 206, "Pune", 950.0, "F")
]


df = spark.createDataFrame(data, schema)

print("=== STEP 1: SOURCE DATA ===")

df.show(truncate=False)

print("\nSchema:")

df.printSchema()

source_count = df.count()

print(f"\nSource row count: {source_count}")

print()


In [0]:
# ------------------------------------------------------------
# Step 2 — SQL NULL Comparison Semantics
#
# NULL represents an unknown / missing value.
#
# Standard SQL equality:
#
# NULL = NULL
#
# does NOT return True.
#
# It returns NULL / UNKNOWN.
# ------------------------------------------------------------

print("=== STEP 2: NULL COMPARISON SEMANTICS ===")

df_null_logic = (
    spark.range(1)
    
    .select(
        F.expr("NULL = NULL").alias("standard_equality"),

        F.expr("NULL <=> NULL").alias("null_safe_equality")))

print("Standard equality vs null-safe equality:")

df_null_logic.show(truncate=False)


print("""
Expected:

NULL = NULL
→ NULL / UNKNOWN

NULL <=> NULL
→ TRUE

<=> is Spark SQL's null-safe equality operator.
""")


In [0]:
# ------------------------------------------------------------
# Step 3 — Why == None Should Not Be Used for NULL Checking
#
# This expression:
#
# F.col("city") == None
#
# translates conceptually to:
#
# city = NULL
#
# Standard SQL comparison with NULL does not produce the
# True/False NULL-checking behaviour we want.
#
# Use:
#
# isNull()
# isNotNull()
#
# instead.
# ------------------------------------------------------------

print(
    "=== STEP 3: isNull() AND isNotNull() ==="
)

df_null_compare = df.select(

    F.col("order_id"),

    F.col("city"),

    (F.col("city") == None).alias("city_equals_none"),

    F.col("city").isNull().alias("city_is_null"),

    F.col("city").isNotNull().alias("city_is_not_null"))

df_null_compare.show(truncate=False)

print("""
Important:

city == None
→ Do NOT use this to test for NULL.

city.isNull()
→ TRUE when city is NULL.

city.isNotNull()
→ TRUE when city contains a non-NULL value.
""")


In [0]:
# ------------------------------------------------------------
# Step 4 — Null-Safe Equality
#
# Sometimes we actually need to compare values while treating
# two NULL values as equal.
#
# Spark provides:
#
# Column.eqNullSafe()
#
# SQL equivalent:
#
# <=>
# ------------------------------------------------------------

print("=== STEP 4: NULL-SAFE EQUALITY ===")


df_null_safe = df.select(

    "order_id",

    "city",

    F.col("city").eqNullSafe(
        F.lit(None)).alias("city_is_null_using_null_safe_equality"))

df_null_safe.show(truncate=False)


print("""
Important distinction:

isNull()
→ Best choice when the requirement is simply:
  "Is this value NULL?"

eqNullSafe() / <=>
→ Useful when comparing two expressions while treating
  matching NULL values as equal.
""")


In [0]:
# ------------------------------------------------------------
# Step 5 — Filter Using isNull() and isNotNull()
# ------------------------------------------------------------

print("=== STEP 5: FILTER NULL / NON-NULL RECORDS ===")

df_nulls = df.filter(F.col("city").isNull())

df_not_nulls = df.filter(F.col("city").isNotNull())

null_city_count = df_nulls.count()

not_null_city_count = df_not_nulls.count()

print(f"Rows where city IS NULL  : " f"{null_city_count}")

df_nulls.show(truncate=False)

print(f"Rows where city IS NOT NULL : " f"{not_null_city_count}")

df_not_nulls.show(truncate=False)

# Validation

assert null_city_count == 2, ("Unexpected city NULL count")

assert not_null_city_count == 4, ("Unexpected city non-NULL count")

assert (null_city_count + not_null_city_count == source_count), "NULL classification did not reconcile"

print("NULL / non-NULL classification validated.")

print()

In [0]:
# ------------------------------------------------------------
# Step 6 — fillna(): Replace NULL Values
#
# fillna() replaces NULL values with supplied defaults.
#
# Important:
#
# A default value should represent a valid business rule.
#
# Do NOT automatically replace every NULL simply because
# NULL exists.
# ------------------------------------------------------------

print("=== STEP 6: fillna() ===")


# ------------------------------------------------------------
# Example 1 — Fill selected STRING columns
# ------------------------------------------------------------

df_filled_str = df.fillna("Unknown", subset=["city", "status"])

print("\nFill city/status NULLs with 'Unknown':")

df_filled_str.show(truncate=False)

# ------------------------------------------------------------
# Example 2 — Fill numeric column
# ------------------------------------------------------------

df_filled_num = df.fillna(0.0, subset=["total_price"])

print("\nFill total_price NULL with 0.0:")

df_filled_num.show(truncate=False)

# ------------------------------------------------------------
# Example 3 — Different Defaults for Different Columns
#
# Dictionary syntax allows us to specify a separate
# replacement value for each column.
#
# NOTE:
# customer_id = -1 is only a learning example.
#
# Whether -1 is an acceptable unknown customer identifier
# depends on the actual business/data-model contract.
# ------------------------------------------------------------

df_filled_all = df.fillna({"city":"Unknown", "status": "Pending", "total_price": 0.0, "customer_id": -1})

print("\nDifferent defaults by column:")

df_filled_all.show(truncate=False)

print("""
Important:

fillna() should be driven by business meaning.

Examples:

city = NULL
→ Maybe "Unknown" is acceptable.

total_price = NULL
→ Replacing with 0 may be WRONG if NULL means
  "price not received".

customer_id = NULL
→ -1 may be a designed unknown-member key,
  but only if the data model defines it that way.
""")

In [0]:
# ------------------------------------------------------------
# Step 7 — dropna(): Remove Rows Containing NULL Values
#
# dropna() removes records according to a NULL rule.
#
# Common options:
#
# how="any"
# → drop row when ANY considered column is NULL.
#
# how="all"
# → drop row only when ALL considered columns are NULL.
#
# subset=[...]
# → evaluate only selected columns.
# ------------------------------------------------------------

print("=== STEP 7: dropna() ===")

# ------------------------------------------------------------
# A. Drop rows where ANY column is NULL
# ------------------------------------------------------------

df_drop_any = df.dropna(how="any")

drop_any_count = df_drop_any.count()

print(
    f"\nRows remaining after how='any': "
    f"{drop_any_count}"
)

df_drop_any.show(truncate=False)

# Expected:
#
# 1001
# 1006
#
# Only these rows have no NULL values.

assert drop_any_count == 2, ("Unexpected dropna(how='any') count")

# ------------------------------------------------------------
# B. Drop rows where ALL columns are NULL
#
# Important:
#
# order_id is non-nullable and populated for every record.
#
# Therefore no complete row has every column NULL.
#
# As a result:
#
# df.dropna(how="all")
#
# removes NOTHING in this dataset.
# ------------------------------------------------------------

df_drop_all = df.dropna(how="all")

drop_all_count = df_drop_all.count()

print(f"\nRows remaining after how='all' " f"across ALL columns: {drop_all_count}")

assert drop_all_count == source_count, ("Unexpected dropna(how='all') result")

# ------------------------------------------------------------
# C. Drop rows where ALL nullable business attributes are NULL
#
# Row 1005 contains:
#
# customer_id = NULL
# city         = NULL
# total_price  = NULL
# status       = NULL
#
# Therefore it should be removed when those columns are
# evaluated with how="all".
# ------------------------------------------------------------

business_columns = [
    "customer_id",
    "city",
    "total_price",
    "status"
]


df_drop_all_business = df.dropna(how="all", subset=business_columns)

drop_all_business_count = (df_drop_all_business.count())

print(
    "\nRows remaining when ALL business "
    f"attributes are NULL: "
    f"{drop_all_business_count}"
)

df_drop_all_business.show(truncate=False)

assert drop_all_business_count == 5, ("Unexpected business-column dropna count")

# ------------------------------------------------------------
# D. Drop when ANY selected key column is NULL
#
# Because how="any" is the default:
#
# dropna(subset=["customer_id", "city"])
#
# removes rows where:
#
# customer_id IS NULL
#
# OR
#
# city IS NULL
# ------------------------------------------------------------

df_drop_subset = df.dropna(subset=["customer_id", "city"])

drop_subset_count = (df_drop_subset.count())

print(
    "\nRows remaining when customer_id "
    f"AND city must both be non-NULL: "
    f"{drop_subset_count}")

df_drop_subset.show(truncate=False)

assert drop_subset_count == 3, ("Unexpected subset dropna count")

print()


In [0]:
# ------------------------------------------------------------
# Step 8 — coalesce(): First Non-NULL Value
#
# F.coalesce() evaluates expressions from left to right
# and returns the first non-NULL value for each row.
#
# Typical uses:
#
# primary value
#     ↓
# secondary fallback
#     ↓
# final fallback
# ------------------------------------------------------------

print("=== STEP 8: F.coalesce() ===")


data_coalesce = [

    (1, "Delhi", None, None),

    (2, None, "Mumbai", None),

    (3, None, None, "Pune"),

    (4, "Chennai", "Kolkata", "Hyderabad"),

    (5, None, None, None )]


df_coal = spark.createDataFrame(data_coalesce,["id", "city_primary", "city_secondary", "city_fallback" ])


df_with_coalesce = df_coal.withColumn(

    "resolved_city",

    F.coalesce(F.col("city_primary"),

        F.col("city_secondary"),

        F.col("city_fallback"),

        F.lit("No City")))

print("\ncoalesce() — first non-NULL value wins:")

df_with_coalesce.show(truncate=False)

print("""
Expected behaviour:

Row 1
Delhi → primary wins

Row 2
Mumbai → secondary wins

Row 3
Pune → fallback wins

Row 4
Chennai → primary wins because it appears first

Row 5
No City → all source city columns are NULL,
          so the literal fallback wins
""")

In [0]:
# ------------------------------------------------------------
# Step 9 — IMPORTANT:
# F.coalesce() vs DataFrame.coalesce()
#
# These have the same name but completely different purposes.
# ------------------------------------------------------------

print("=== STEP 9: TWO DIFFERENT coalesce OPERATIONS ===")


print("""
F.coalesce(...)

Example:

F.coalesce(
    F.col("primary"),
    F.col("secondary")
)

Purpose:
→ Returns the first non-NULL VALUE.


df.coalesce(4)

Purpose:
→ Reduces the number of DataFrame PARTITIONS.

These are completely different concepts.

S3.10 uses F.coalesce() for NULL handling.
""")

In [0]:
# ------------------------------------------------------------
# Step 10 — Silver-Layer Data Quality Interpretation
# ------------------------------------------------------------

print(
    "=== STEP 10: SILVER-LAYER INTERPRETATION ==="
)


print("""
Possible NULL-handling flow:

Bronze source
      ↓
Identify NULL values
      ↓
Understand business meaning
      ↓
 ┌──────────────────────────────┐
 │                              │
NULL allowed                NULL invalid
 │                              │
Keep                        Validate
 │                              │
 │                   ┌──────────┴──────────┐
 │                   │                     │
 │               Default valid        Record invalid
 │                   │                     │
 │                fillna()           Reject/Quarantine
 │
 ↓
Standardized Silver data


Important:

Do not automatically:

- fill every NULL
- drop every NULL
- replace every numeric NULL with zero
- replace every missing key with -1

The correct action depends on the business contract.
""")


In [0]:
# ------------------------------------------------------------
# Step 11 — Transformations vs Actions
# ------------------------------------------------------------

print(
    "=== STEP 11: TRANSFORMATIONS VS ACTIONS ==="
)


print("""
Transformations used:

select()
filter()
fillna()
dropna()
withColumn()

Column-expression functions used:

isNull()
isNotNull()
eqNullSafe()
coalesce()

Actions used:

show()
count()

Spark transformations remain part of the lazy
execution model until an action requires a result.
""")

In [0]:
# ------------------------------------------------------------
# Step 12 — Final Validation
# ------------------------------------------------------------

print("=== STEP 12: FINAL VALIDATION ===")


assert source_count == 6

assert null_city_count == 2

assert not_null_city_count == 4

assert drop_any_count == 2

assert drop_all_count == 6

assert drop_all_business_count == 5

assert drop_subset_count == 3


print(
    "All NULL-handling validations passed."
)

print()

In [0]:
# ------------------------------------------------------------
# Step 13 — Final Learning Summary
# ------------------------------------------------------------

print("=== FINAL LEARNING SUMMARY ===")


print("""
NULL:

- Represents missing / unknown information.
- Is not the same as zero.
- Is not the same as an empty string.
- Is not the same as False.


Standard equality:

NULL = NULL
→ NULL / UNKNOWN


isNull():

col.isNull()

→ Tests whether a value is NULL.


isNotNull():

col.isNotNull()

→ Tests whether a value is not NULL.


Null-safe equality:

col.eqNullSafe(other)

or SQL:

<=>

→ Allows NULL-safe comparison.


fillna():

→ Replaces NULL values with defined defaults.

Use only when the replacement has valid
business meaning.


dropna():

→ Removes records according to NULL conditions.

how="any"
→ Drop when any considered value is NULL.

how="all"
→ Drop when all considered values are NULL.

subset=[...]
→ Restrict NULL evaluation to selected columns.


F.coalesce():

→ Returns the first non-NULL value from a sequence
  of expressions.

Example:

primary
   ↓
secondary
   ↓
fallback
   ↓
default literal


Important:

F.coalesce(...)
→ NULL-value resolution.

DataFrame.coalesce(n)
→ Partition reduction.

They are completely different.


Silver-layer principle:

Identify NULL
     ↓
Understand its meaning
     ↓
Apply business rule
     ↓
Keep / Fill / Reject / Quarantine
     ↓
Validate
     ↓
Silver-ready data
""")


print("=== S3.10 CELL 2 COMPLETED ===")

# S3.10 — Key Takeaways: Null Handling

## 1. NULL Basics

- `NULL` means missing / unknown information.
- `NULL` is NOT the same as:
  - `0`
  - empty string `""`
  - `False`
- Standard equality does not treat two NULL values as equal.

```text
NULL = NULL
→ NULL / UNKNOWN
```

For NULL checks, use:

```python
F.col("city").isNull()
F.col("city").isNotNull()
```

Do not use:

```python
F.col("city") == None
```

for normal NULL checking.

---

# 2. NULL-Safe Comparison

Spark supports NULL-safe equality.

### PySpark

```python
F.col("city").eqNullSafe(
    F.col("other_city")
)
```

### Spark SQL

```text
<=>
```

Example:

```text
NULL = NULL
→ NULL / UNKNOWN

NULL <=> NULL
→ TRUE
```

Use NULL-safe equality when comparing expressions where both sides
may contain NULL values.

---

# 3. Main NULL Functions

| Function | Purpose |
|---|---|
| `isNull()` | Returns TRUE when the value is NULL |
| `isNotNull()` | Returns TRUE when the value is not NULL |
| `fillna()` | Replaces NULL values with supplied defaults |
| `dropna()` | Removes rows according to NULL rules |
| `F.coalesce()` | Returns the first non-NULL expression |

---

# 4. isNull() and isNotNull()

These return Boolean Spark Column expressions.

They can be used in:

```text
filter()
where()
select()
when()
```

Example:

```python
df.filter(
    F.col("city").isNull()
)
```

or:

```python
df.select(
    F.col("city").isNull().alias("city_is_null")
)
```

---

# 5. fillna()

`fillna()` replaces NULL values.

Example:

```python
df.fillna({
    "city": "Unknown",
    "status": "Pending"
})
```

### Important Rule

Only replace NULL when the replacement has valid business meaning.

Example:

```text
NULL city
→ "Unknown"
```

may be acceptable if the business model defines `"Unknown"` as a valid
fallback value.

But:

```text
NULL total_price
→ 0.0
```

may be misleading if NULL means:

```text
price not received
price unavailable
price not yet calculated
```

Zero and NULL have different meanings.

### Principle

```text
NULL
↓
Understand business meaning
↓
Choose whether to:
Keep / Fill / Reject / Quarantine
```

---

# 6. dropna()

`dropna()` removes rows based on NULL conditions.

## how="any"

```python
df.dropna(
    how="any"
)
```

Drop the row when ANY considered column is NULL.

---

## how="all"

```python
df.dropna(
    how="all"
)
```

Drop the row only when ALL considered columns are NULL.

---

## subset

Use `subset` to limit the NULL check to specific columns.

Example:

```python
df.dropna(
    subset=[
        "customer_id",
        "city"
    ]
)
```

This means only:

```text
customer_id
city
```

are considered for the NULL rule.

---

## how + subset

Example:

```python
df.dropna(
    how="all",
    subset=[
        "customer_id",
        "city",
        "status"
    ]
)
```

The row is removed only when all three selected columns are NULL.

### Important

```text
how
→ defines ANY vs ALL

subset
→ defines which columns are checked
```

---

# 7. F.coalesce()

`F.coalesce()` returns the first non-NULL expression from left to right.

Example:

```python
F.coalesce(
    F.col("city_primary"),
    F.col("city_secondary"),
    F.col("city_fallback"),
    F.lit("No City")
)
```

Conceptually:

```text
city_primary
     ↓
if NULL

city_secondary
     ↓
if NULL

city_fallback
     ↓
if NULL

"No City"
```

The first non-NULL value wins.

---

# 8. Critical: Two Different coalesce Operations

These have the same name but completely different purposes.

## F.coalesce()

```python
F.coalesce(
    F.col("col1"),
    F.col("col2")
)
```

Purpose:

```text
NULL-value resolution
```

Returns the first non-NULL value.

---

## DataFrame.coalesce()

```python
df.coalesce(4)
```

Purpose:

```text
Partition reduction
```

Reduces the number of DataFrame partitions.

### Easy way to remember

```text
F.coalesce(...)
→ VALUES

df.coalesce(n)
→ PARTITIONS
```

---

# 9. NULL Is Not a Data-Quality Error by Itself

NULL does not automatically mean the data is bad.

Examples:

```text
middle_name = NULL
→ may be valid

delivery_date = NULL
→ may be valid for an open order

customer_id = NULL
→ may be invalid if customer_id is mandatory
```

The business contract determines whether a NULL is acceptable.

---

# 10. Silver-Layer NULL Decision Pattern

```text
Bronze Source
     ↓
Identify NULL
     ↓
Understand business meaning
     ↓
Is NULL allowed?
     ↓
 ┌───────────────┬────────────────────┐
 │               │                    │
Yes           Default valid       Invalid
 │               │                    │
Keep           fillna()         Reject / Quarantine
 │               │                    │
 └───────────────┴────────────────────┘
                 ↓
              Validate
                 ↓
         Silver-ready data
```

Do not automatically:

```text
fill every NULL
drop every NULL
replace every numeric NULL with zero
replace every missing key with -1
```

---

# 11. Quick Decision Guide

| Requirement | Function |
|---|---|
| Check whether value is NULL | `isNull()` |
| Check whether value is present | `isNotNull()` |
| Compare values NULL-safely | `eqNullSafe()` / `<=>` |
| Replace NULL with default | `fillna()` |
| Remove rows containing NULL | `dropna()` |
| Select first available value | `F.coalesce()` |

---

# 12. Final S3.10 Understanding

```text
Need to check NULL?
→ isNull()

Need to check non-NULL?
→ isNotNull()

Need NULL-safe comparison?
→ eqNullSafe() / <=>

Need default replacement?
→ fillna()

Need to remove rows based on NULL rules?
→ dropna()

Need first available value?
→ F.coalesce()
```

## Core Principle

```text
Identify NULL
     ↓
Understand meaning
     ↓
Apply business rule
     ↓
Keep / Fill / Reject / Quarantine
     ↓
Validate
     ↓
Silver
```

**S3.10 — NULL Handling completed.**